# 10 — LightGBM Forecasting

Repeat the same experimental design with LightGBM so the comparison is controlled.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import pandas as pd
from src.utils.config import load_yaml
from src.features.feature_pipeline import MODEL_FEATURES
from src.forecasting.lightgbm_model import build_lightgbm
from src.forecasting.evaluate import evaluate
df=pd.read_parquet(ROOT/"data/processed/forecast_features.parquet"); cfg=load_yaml("model_config.yaml"); maxd=df.date.max(); test_start=maxd-pd.Timedelta(days=cfg["test_days"]-1); val_start=test_start-pd.Timedelta(days=cfg["validation_days"]); train=df[df.date<val_start]; val=df[(df.date>=val_start)&(df.date<test_start)]
model=build_lightgbm(cfg["lightgbm"],cfg["random_seed"],cfg["n_jobs"]); model.fit(train[MODEL_FEATURES],train.demand); pred=model.predict(val[MODEL_FEATURES]).clip(min=0); print(evaluate(val.demand,pred))

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
imp=pd.Series(model.feature_importances_,index=MODEL_FEATURES).sort_values(ascending=False).head(15); fig,ax=plt.subplots(figsize=(8,5)); imp.sort_values().plot(kind="barh",ax=ax); ax.set_title("LightGBM feature importance"); plt.show()

The key comparison is not which library is fashionable; it is whether performance improves consistently under the same chronological split and metrics.